# h5ad Structure Inspection: ovary_1 vs ovary_2

Compare object attributes between ovary_1 (manually curated) and ovary_2 (CellxGene) to identify structural differences that may cause integration issues.

In [ ]:
# Locate repository root and import path configuration
import os
import sys
from pathlib import Path

def _find_repo():
    env = os.environ.get("ATLAS_PIPELINE_ROOT", "").strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "pipeline_paths.py").is_file():
            return p
    cwd = Path.cwd().resolve()
    for d in (cwd, cwd.parent):
        if (d / "pipeline_paths.py").is_file():
            return d
    raise RuntimeError(
        "Set ATLAS_PIPELINE_ROOT to the repository root (folder containing pipeline_paths.py), "
        "or start Jupyter from that directory."
    )

REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import pipeline_paths as pp


In [ ]:
import os
import scanpy as sc
import numpy as np
import pandas as pd

INPUT_DIR = str(pp.PROCESSED_H5AD)

# Load both objects
ovary_1 = sc.read_h5ad(os.path.join(INPUT_DIR, "ovary_1.h5ad"))
ovary_2 = sc.read_h5ad(os.path.join(INPUT_DIR, "ovary_2.h5ad"))

print("Loaded ovary_1 and ovary_2")

## Basic dimensions

In [ ]:
print("\n=== Dimensions ===")
print(f"ovary_1: {ovary_1.n_obs:,} cells × {ovary_1.n_vars:,} genes")
print(f"ovary_2: {ovary_2.n_obs:,} cells × {ovary_2.n_vars:,} genes")

## adata.X: type, dtype, sparse vs dense

In [ ]:
print("=== adata.X ===")
print("\novary_1:")
print(f"  type: {type(ovary_1.X)}")
print(f"  dtype: {ovary_1.X.dtype}")
print(f"  shape: {ovary_1.X.shape}")
if hasattr(ovary_1.X, 'format'):
    print(f"  sparse format: {ovary_1.X.format}")

print("\novary_2:")
print(f"  type: {type(ovary_2.X)}")
print(f"  dtype: {ovary_2.X.dtype}")
print(f"  shape: {ovary_2.X.shape}")
if hasattr(ovary_2.X, 'format'):
    print(f"  sparse format: {ovary_2.X.format}")

## Counts: integers vs floats

In [ ]:
def check_int_or_float(arr, name):
    """Sample values to check if data looks like counts (ints) or normalized (floats)"""
    if hasattr(arr, 'toarray'):
        sample = arr[:10, :100].toarray()
    else:
        sample = np.asarray(arr)[:10, :100]
    flat = sample.flatten()
    flat = flat[flat != 0]
    if len(flat) == 0:
        return "(all zeros in sample)"
    is_int_like = np.allclose(flat, np.round(flat))
    print(f"{name}: dtype={arr.dtype}, values look like integers: {is_int_like}")
    if len(flat) > 0:
        print(f"  sample non-zero values: {flat[:5]}")

print("=== X matrix ===")
check_int_or_float(ovary_1.X, "ovary_1.X")
check_int_or_float(ovary_2.X, "ovary_2.X")

print("\n=== layers (if 'counts' exists) ===")
for name, adata in [("ovary_1", ovary_1), ("ovary_2", ovary_2)]:
    if "counts" in adata.layers:
        check_int_or_float(adata.layers["counts"], f"{name}.layers['counts']")
    else:
        print(f"{name}: no 'counts' layer")

## .raw attribute

In [ ]:
def print_raw_nonzero(adata, name):
    """Print .raw info and sample of nonzero values."""
    if adata.raw is None:
        print(f"{name}: no .raw")
        return
    X = adata.raw.X
    nz = X.nnz if hasattr(X, 'nnz') else int(np.count_nonzero(X))
    print(f"{name}.raw: shape={adata.raw.shape}, dtype={X.dtype}")
    print(f"  Nonzero entries: {nz:,} (total elements: {np.prod(adata.raw.shape):,})")
    # Sample nonzero values (first few)
    if hasattr(X, 'data'):
        sample = X.data[:10]
    else:
        sample = X[X != 0].flatten()[:10]
    if len(sample) > 0:
        print(f"  Sample nonzero values: {sample}")

print("=== .raw attribute ===")
print_raw_nonzero(ovary_1, "ovary_1")
print()
print_raw_nonzero(ovary_2, "ovary_2")

## Layers

In [ ]:
print("=== .layers ===")
print(f"ovary_1 layers: {list(ovary_1.layers.keys())}")
print(f"ovary_2 layers: {list(ovary_2.layers.keys())}")

## obs columns

In [ ]:
print("=== .obs columns ===")
o1_cols = set(ovary_1.obs.columns)
o2_cols = set(ovary_2.obs.columns)
print(f"ovary_1: {sorted(o1_cols)}")
print(f"\novary_2: {sorted(o2_cols)}")
print(f"\nOnly in ovary_1: {sorted(o1_cols - o2_cols)}")
print(f"Only in ovary_2: {sorted(o2_cols - o1_cols)}")
print(f"In both: {sorted(o1_cols & o2_cols)}")

print(ovary_2.obs[['tissue', 'tissue_type']].head())

## obsm (embeddings)

In [ ]:
print("=== .obsm ===")
print(f"ovary_1: {list(ovary_1.obsm.keys())}")
print(f"ovary_2: {list(ovary_2.obsm.keys())}")

## uns (unstructured metadata)

In [ ]:
print("=== .uns (top-level keys) ===")
print(f"ovary_1: {list(ovary_1.uns.keys())}")
print(f"ovary_2: {list(ovary_2.uns.keys())}")

## var (gene info)

In [ ]:
print("=== .var columns ===")
print(f"ovary_1 var cols: {list(ovary_1.var.columns)}")
print(f"ovary_2 var cols: {list(ovary_2.var.columns)}")

# Gene naming: are they using same gene IDs?
print("\n=== Gene index (first 5) ===")
print(f"ovary_1: {list(ovary_1.var.index[:5])}")
print(f"ovary_2: {list(ovary_2.var.index[:5])}")

overlap = set(ovary_1.var.index) & set(ovary_2.var.index)
print(f"\nGene overlap: {len(overlap):,} genes in both")

## Summary comparison table

In [ ]:
summary = pd.DataFrame({
    "ovary_1": {
        "n_obs": ovary_1.n_obs,
        "n_vars": ovary_1.n_vars,
        "X type": type(ovary_1.X).__name__,
        "X dtype": str(ovary_1.X.dtype),
        "has .raw": ovary_1.raw is not None,
        "layers": list(ovary_1.layers.keys()),
    },
    "ovary_2": {
        "n_obs": ovary_2.n_obs,
        "n_vars": ovary_2.n_vars,
        "X type": type(ovary_2.X).__name__,
        "X dtype": str(ovary_2.X.dtype),
        "has .raw": ovary_2.raw is not None,
        "layers": list(ovary_2.layers.keys()),
    },
}).T
display(summary)

## Revise ovary_1.h5ad in the processed folder to match CellXGene h5ad downloads

In [ ]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import mygene

INPUT_DIR = str(pp.PROCESSED_H5AD)

# Load both objects
adata = sc.read_h5ad(os.path.join(INPUT_DIR, "ovary_1.h5ad"))

# -------------------------
# 1. Convert matrix format
# -------------------------
adata.X = adata.X.tocsr().astype(np.float32)

# Save raw counts
raw_counts = adata.X.copy()

# -------------------------
# 2. Normalize like CellxGene datasets
# -------------------------
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.X = adata.X.astype(np.float32)

# -------------------------
# 3. Convert gene symbols → ENSG IDs (before creating .raw so we subset consistently)
# -------------------------
symbols = adata.var_names.tolist()

mg = mygene.MyGeneInfo()
query = mg.querymany(symbols,
                     scopes="symbol",
                     fields="ensembl.gene",
                     species="human")

mapping = {}
for q in query:
    if "ensembl" in q:
        ens = q["ensembl"]
        if isinstance(ens, list):
            mapping[q["query"]] = ens[0]["gene"]
        else:
            mapping[q["query"]] = ens["gene"]

adata.var["feature_name"] = adata.var_names
adata.var["ensembl_id"] = adata.var_names.map(mapping)

# keep only genes with ENSG IDs; subset raw_counts to same genes
keep_mask = adata.var["ensembl_id"].notnull().values
raw_counts = raw_counts[:, keep_mask]
adata = adata[:, keep_mask].copy()

adata.var_names = adata.var["ensembl_id"]
adata.var_names_make_unique()

# -------------------------
# 4. Create .raw attribute (raw counts for ENSG-subset genes)
# -------------------------
adata.raw = sc.AnnData(X=raw_counts.astype(np.float32), var=adata.var.copy())

# -------------------------
# 5. Add missing .var columns
# -------------------------
required_cols = [
    "feature_is_filtered",
    "feature_name",
    "feature_reference",
    "feature_biotype",
    "feature_length",
    "feature_type"
]

for col in required_cols:
    if col not in adata.var:
        adata.var[col] = np.nan

# -------------------------
# 6. Add requested obs columns (match ovary_2/CellxGene naming)
# -------------------------
adata.obs["tissue"] = "ovary"
adata.obs["tissue_id"] = "ovary_1"  # required for harmonization key merge
adata.obs["development_stage"] = adata.obs["Age"].astype(str)

# -------------------------
# 7. Ensure CSR format everywhere; clear layers to match CellxGene
# -------------------------
adata.X = adata.X.tocsr().astype(np.float32)
# Raw.X is read-only; replace with new AnnData using CSR matrix
adata.raw = sc.AnnData(
    X=adata.raw.X.tocsr().astype(np.float32),
    var=adata.raw.var.copy(),
)
for k in list(adata.layers.keys()):
    del adata.layers[k]

# -------------------------
# 8. Save fixed object
# -------------------------
adata.write_h5ad(os.path.join(INPUT_DIR, "ovary_1.h5ad"))
print(f"Saved revised ovary_1.h5ad: {adata.n_obs} cells × {adata.n_vars} genes")


## Check that the new ovary_1 file matches ovary_2

In [ ]:
# adata holds the revised ovary_1 (from previous cell); ovary_2 is the CellxGene reference

# Gene overlap (both use ENSG IDs after revision)
gene_overlap = len(set(adata.var_names) & set(ovary_2.var_names))
print("=== Check: revised ovary_1 vs ovary_2 ===\n")
print(f"Gene overlap (both ENSG): {gene_overlap:,} genes")

# Key structural checks
print(f"\nStructural alignment:")
print(f"  Both have .raw: ovary_1={adata.raw is not None}, ovary_2={ovary_2.raw is not None}")
print(f"  X dtype: ovary_1={adata.X.dtype}, ovary_2={ovary_2.X.dtype}")
print(f"  X format: ovary_1={type(adata.X).__name__}, ovary_2={type(ovary_2.X).__name__}")
print(f"  layers: ovary_1={list(adata.layers.keys())}, ovary_2={list(ovary_2.layers.keys())}")

# Obs columns needed for integration (tissue_id required for harmonization key)
for col in ["cell_type", "donor_id", "tissue", "tissue_id"]:
    in1 = col in adata.obs.columns
    in2 = col in ovary_2.obs.columns
    status = "✓" if (in1 and in2) else ("⚠" if in1 or in2 else "✗")
    print(f"  obs['{col}']: ovary_1={in1}, ovary_2={in2} {status}")

# Sample of nonzero values in X (compare normalization: counts = integers; normalized = floats)
print("\nX nonzero value samples:")
for name, obj in [("adata (ovary_1 revised)", adata), ("ovary_2", ovary_2)]:
    X = obj.X
    nz = X.nnz if hasattr(X, 'nnz') else int(np.count_nonzero(X))
    sample = X.data[:15] if hasattr(X, 'data') else X[X != 0].flatten()[:15]
    print(f"  {name}: {nz:,} nonzero, sample={sample}")

# Sample of nonzero values in .raw (expect counts = integers; normalized = floats)
print("\n.raw nonzero value samples:")
for name, obj in [("adata (ovary_1 revised)", adata), ("ovary_2", ovary_2)]:
    if obj.raw is not None:
        X = obj.raw.X
        nz = X.nnz if hasattr(X, 'nnz') else int(np.count_nonzero(X))
        sample = X.data[:15] if hasattr(X, 'data') else X[X != 0].flatten()[:15]
        print(f"  {name}: {nz:,} nonzero, sample={sample}")
    else:
        print(f"  {name}: no .raw")

summary = pd.DataFrame({
    "ovary_1 (revised)": {
        "n_obs": adata.n_obs,
        "n_vars": adata.n_vars,
        "X type": type(adata.X).__name__,
        "X dtype": str(adata.X.dtype),
        "has .raw": adata.raw is not None,
        "layers": list(adata.layers.keys()),
    },
    "ovary_2": {
        "n_obs": ovary_2.n_obs,
        "n_vars": ovary_2.n_vars,
        "X type": type(ovary_2.X).__name__,
        "X dtype": str(ovary_2.X.dtype),
        "has .raw": ovary_2.raw is not None,
        "layers": list(ovary_2.layers.keys()),
    },
}).T
print("\nSummary table:")
display(summary)